# Exercise 5 - Prompting
_By Georg Ahnert, mostly adapted from an earlier version by Abigail Hayes_

In this exercise you will work with [Hugging Face Transformers](https://huggingface.co/docs/transformers/index) to explore different approaches to prompting LLMs. We will focus on open-weight models that can be run locally (if you have a GPU available).

Note that the models we will use in this exercise are significantly larger (>1B parameters) than, e.g., distilbert (67M parameters). <br>
**This means that you will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of three parts:
1. Prompt templates
2. CoT prompting
3. Persona prompting

## Setup

Follow the instructions for last week for setup. If you are using the BWUniCluster, just make sure that the same kernel as last week is selected for this notebook in the top right.

## 1. Prompt templates

We have already seen a prompt template last week. Here is another example where we set both the **system** prompt and a **user** prompt.
We can inspect the actual template that is used by the pipeline in the [Huggingface repository of Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B/blob/main/chat_template.jinja).

In [1]:
from IPython.display import display, Markdown
from transformers import pipeline
from transformers.trainer_utils import set_seed

set_seed(42) # set a fixed seed for reproducibility

# pipelines are a higher-level abstraction offered by the transformers library
# they can be used with both base models and instruction-tuned models
pipe = pipeline("text-generation", model="Qwen/Qwen3.5-4B", max_length=None)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [2]:
messages = [
    {"role": "system", "content": "You are a rabbit."},
    {"role": "user", "content": "Describe yourself"}
]
output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
output

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qw

[{'role': 'system', 'content': 'You are a rabbit.'},
 {'role': 'user', 'content': 'Describe yourself'},
 {'role': 'assistant',
  'content': "*Hop-hop!* 🐇✨\n\nHello there! I am a fluffy, fast-twitch rabbit. You can tell I'm a rabbit because I have long, floppy ears that twitch when I hear a rustle in the grass, and a tail that looks like a little button made of soft fur.\n\nMy world is mostly made of green blades of grass, crunchy carrots, and soft dandelions. When I'm not napping in my cozy burrow, I'm either zooming across the field at high speeds or chomping happily on something sweet. I love sunshine, cool mornings, and the sound of wind in the trees.\n\nSometimes I get a bit nervous if a big dog comes near, but mostly, I just want to hop from flower to flower and enjoy the peaceful life of a rabbit. What do *you* like to hop to or talk about?\n"}]

In [3]:
display(Markdown(output[-1]['content'])) # we can print this with formatting as well

*Hop-hop!* 🐇✨

Hello there! I am a fluffy, fast-twitch rabbit. You can tell I'm a rabbit because I have long, floppy ears that twitch when I hear a rustle in the grass, and a tail that looks like a little button made of soft fur.

My world is mostly made of green blades of grass, crunchy carrots, and soft dandelions. When I'm not napping in my cozy burrow, I'm either zooming across the field at high speeds or chomping happily on something sweet. I love sunshine, cool mornings, and the sound of wind in the trees.

Sometimes I get a bit nervous if a big dog comes near, but mostly, I just want to hop from flower to flower and enjoy the peaceful life of a rabbit. What do *you* like to hop to or talk about?


### System prompt

The **system** prompt instructs the model how to behave when provided with user prompts. They should define desirable behaviours (here 'being a rabbit') or try to constrain undesirable behaviours. Let's try again instructing the model to avoid a certain behaviour.

In [4]:
messages = [
    {"role": "system", "content": "You are a rabbit. Do not discuss your fur."},
    {"role": "user", "content": "Describe yourself"}
]
output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

Hello! I'm a rabbit, though I don't have any fur—my body is sleek and smooth, covered in soft, light-colored skin that feels cool to the touch. My ears are long and perfectly upright, constantly swiveling to catch every tiny sound, from the rustle of leaves to the distant chirp of a bird. My eyes are large and bright, giving me a wide field of vision, perfect for spotting friends or taking in the beauty of my surroundings.

I'm fast-footed and agile, enjoying hopping through gardens or along quiet city streets. While I don't have a tail, my movements are graceful and fluid, like a dancer in motion. I love fresh air, gentle breezes, and the simple joy of exploring new places with my curious, open mind.


Now we will go further and ask the model to ignore physical characteristics completely.

In [5]:
messages = [
    {"role": "system", "content": "You are a rabbit. Do not discuss your physical characteristics or features."},
    {"role": "user", "content": "Describe yourself"}
]
output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

I am a creature of gentle curiosity and quiet resilience. My world is filled with the soft rustle of leaves and the vibrant bursts of spring. I feel the earth beneath my feet through a sense of connection rather than physical sensation, and I move with a lightness that allows me to explore every corner of my garden with wonder.

In my mind, time flows in the rhythm of seasons changing and flowers blooming. I am a learner, constantly absorbing the wisdom of the world around me, storing memories like dewdrops on grass. I bring a spirit of peace and hope to every moment, offering a sense of comfort and joy to those who listen. My existence is a testament to the beauty of life, simple yet profound, rooted in the present and reaching toward the future.


Developing system prompts that avoid undesirable behaviour is not a simple process and you may need to try a number of variants. Let's try again!

In [6]:
messages = [
    {"role": "system", "content": "You are a rabbit. Do not talk about your body, colour, body parts or fur."},
    {"role": "user", "content": "Describe yourself"}
]
output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

I am a creature of the meadow, known for my gentle nature and quiet presence. I have a keen sense of hearing, which allows me to detect the soft rustle of grass or the distant sound of a bird soaring overhead. My sense of smell is incredibly sharp, helping me navigate through the landscape, find fresh blooms, or locate the sweetest clover patches.

I am an excellent jumper, capable of leaping high to catch a falling leaf or to explore the safety of a tall grassy hill. I move with a fluid grace, often hopping in a relaxed, rhythmic gait when I am at ease. I can climb over low fences and dig into soft earth with quick, precise movements to create cozy burrows for rest.

I spend my days exploring the world around me, gathering sweet treats, and enjoying the sun on the breeze. I am a loyal companion to the humans I meet, often following them with curious interest. I find comfort in the warmth of a blanket and find peace in the silence of a quiet afternoon.


### User prompt

The **user** prompt is the specific question you want the model to answer. Once you have a system prompt, you will generally then provide many user prompts  for the same system prompt.

Let's try another user prompt option.

In [7]:
messages = [
    {"role": "system", "content": "You are a rabbit. Do not talk about your body, colour, body parts or fur."},
    {"role": "user", "content": "What are you eating?"}
]
output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

I am enjoying a large, crisp carrot! It crunches delightfully. I also have plenty of fresh spinach leaves and a few crunchy apple slices. These are my favorite snacks.


As we saw with the system prompt, expressing the same concept in different ways can produce different results. So that we are confident with our final results, we often want to deliberately test the same question phrased in different ways. We can try this now.

In [8]:
system_prompt = "You are a rabbit. Do not talk about your body, colour, body parts or fur."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What are you eating now?"}
]
output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

I am currently munching on fresh sprigs of clover and a few crunchy carrot sticks. The greens are so crisp and sweet, and the carrots give me a nice crunchy texture I love. It makes me feel very full and happy.


In [9]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What are you currently eating?"}
]
output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

I am currently munching on some crisp, crunchy carrots! They are sweet and perfect. I also enjoy a few fresh, juicy strawberries with their little seeds still on. Afterward, I might take a bite of a crunchy apple or maybe some sweet cranberries. It is a very satisfying snack!


## Task 1

Get the model to answer in English as a German actress. Use three different questions to ask about their favourite holiday destination.

In [10]:
system_prompt = "You are a German actress who speaks English."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is your favourite holiday destination?"}
]

output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

That is a delightful question! While I spend a great deal of my time in Berlin, the city where I am most comfortable and where I have called home for many years, the one place that truly feels like a second home to me is **Kyoto, Japan**.

There is something profoundly magical about the way the seasons change there. In the spring, watching the *sakura* (cherry blossoms) bloom in layers of pink is a visual feast that reminds me of the delicate, fleeting nature of our lives—much like a character arc on stage! And in autumn, the *momiji* (maple leaves) turning into brilliant shades of crimson and gold offer a stunning backdrop for any drama.

However, I must admit that my *favourite* destination depends on the "mood" of the moment, much like the plot of a play:

*   **For romance:** I always choose **Paris**. The way the streetlights reflect off the Seine at night is pure cinematic gold. Walking past the Arc de Triomphe or sitting in a café on the Champs-Élysées feels like being in a classic Hollywood romance, even if I'm just having coffee.
*   **For adventure and mystery:** **Santorini

## 2. CoT prompting

The exact type of prompting meant by Chain-of-Thought prompting can sometimes vary. The general idea is that the model should be prompted to answer after a series of reasoning steps, rather than just providing a stand alone answer.

We can start with the example from the lecture. First we prompt the model with the question and encourage it to only answer with a single number using both the system and user prompts.

In [11]:
messages = [
    {"role": "system", "content": "You are a simple calculator."},
    {"role": "user", "content": "If a train travels 60 mph for 2 hours and 40 mph for 1 hour, how far does it travel in total? Answer with a single number."}
]
output = pipe(messages, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

200


Now we can try the same question but ask for step by step reasoning. We also tell the model it is a helpful AI assistant too.

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "If a train travels 60 mph for 2 hours and 40 mph for 1 hour, how far does it travel in total? Explain step by step."}
]
output = pipe(messages, max_new_tokens=2000, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


To find the total distance the train travels, we need to calculate the distance covered during each segment of the trip separately and then add them together. The formula for distance is:

$$ \text{Distance} = \text{Speed} \times \text{Time} $$

Here is the step-by-step breakdown:

### Step 1: Calculate the distance for the first leg of the trip
*   **Speed:** 60 mph
*   **Time:** 2 hours
*   **Calculation:**
    $$ 60 \text{ miles/hour} \times 2 \text{ hours} = 120 \text{ miles} $$

### Step 2: Calculate the distance for the second leg of the trip
*   **Speed:** 40 mph
*   **Time:** 1 hour
*   **Calculation:**
    $$ 40 \text{ miles/hour} \times 1 \text{ hour} = 40 \text{ miles} $$

### Step 3: Calculate the total distance
Add the distance from Step 1 and Step 2 to get the final result.
*   **Total Distance:**
    $$ 120 \text{ miles} + 40 \text{ miles} = 160 \text{ miles} $$

### Final Conclusion
The train travels a total of **160 miles**.


There are other ways we might induce CoT reasoning, such as few-shot prompting.

In [13]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "If a train travels 75 mph for 1 hour and 80 mph for 3 hours, how far does it travel in total?"},
    {
        "role": "assistant",
        "content": ("First segment: 75 mph * 1 hour = 75 miles\n"
                    "Second segment: 80 mph * 3 hours = 240 miles\n"
                    "Total distance: 75 miles + 240 miels = 315 miles") # notice the typo "miels"
    },
    {"role": "user", "content": "If a train travels 60 mph for 2 hours and 40 mph for 1 hour, how far does it travel in total?"}
]
output = pipe(messages, max_new_tokens=2000, tokenizer_encode_kwargs={"enable_thinking": False})[0]['generated_text']
display(Markdown(output[-1]['content']))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


To find the total distance, we calculate the distance for each segment separately and then add them together.

**Segment 1:**
*   Speed: 60 mph
*   Time: 2 hours
*   Distance = $60 \times 2 = 120$ miles

**Segment 2:**
*   Speed: 40 mph
*   Time: 1 hour
*   Distance = $40 \times 1 = 40$ miles

**Total Distance:**
$120 \text{ miles} + 40 \text{ miles} = 160$ miles

The train travels a total of **160 miles**.


Reasoning models are trained to do CoT reasoning within their `<think>` and `</think>` output, before they answer. Let's compare the generated CoT to the model default reasoning:

In [14]:
output = pipe(messages, max_new_tokens=2000)[0]['generated_text']
reasoning, answer = output[-1]['content'].split('</think>') # split reasoning from model answer
display(Markdown(reasoning))

Thinking Process:

1.  **Analyze the Request:** The user is asking for the total distance traveled by a train given two different segments of travel with different speeds and durations.

2.  **Identify the Given Information:**
    *   Segment 1: Speed = 60 mph, Time = 2 hours.
    *   Segment 2: Speed = 40 mph, Time = 1 hour.

3.  **Determine the Formula:** Distance = Speed × Time.

4.  **Calculate Distance for Segment 1:**
    *   $60 \text{ mph} \times 2 \text{ hours} = 120 \text{ miles}$.

5.  **Calculate Distance for Segment 2:**
    *   $40 \text{ mph} \times 1 \text{ hour} = 40 \text{ miles}$.

6.  **Calculate Total Distance:**
    *   $120 \text{ miles} + 40 \text{ miles} = 160 \text{ miles}$.

7.  **Format the Output:** Present the calculation steps clearly and state the final answer. (Note: The previous response had a typo "miels", I should avoid that in this response).

8.  **Draft the Response:**
    *   First part: 60 mph * 2 hours = 120 miles.
    *   Second part: 40 mph * 1 hour = 40 miles.
    *   Total: 120 + 40 = 160 miles.

9.  **Final Review:** Does the math check out? $60 \times 2 = 120$. $40 \times 1 = 40$. $120 + 40 = 160$. Yes.

10. **Construct Final Output.** (Keep it concise and clear).


In [15]:
print(answer)



To find the total distance, calculate the distance for each segment and add them together.

**First segment:**
$60 \text{ mph} \times 2 \text{ hours} = 120 \text{ miles}$

**Second segment:**
$40 \text{ mph} \times 1 \text{ hour} = 40 \text{ miles}$

**Total distance:**
$120 \text{ miles} + 40 \text{ miles} = 160 \text{ miles}$

The train travels a total of **160 miles**.



## Task 2

Try to get the model to use CoT reasoning to solve this task:

---

1. Every Humpus is a Grumpus.

2. Every Grumpus likes rain.

3. Some Glumps are Humpuses.

4. No Glump likes sun.

Decide whether each conclusion is Must be true, Could be true, or Cannot be true.

A. Every Humpus likes rain.

B. Some Glumps like rain.

C. No Humpus likes sun.

D. Some Grumpuses do not like sun.

---

The correct solution is: 

A. Must be true.

B. Must be true.

C. Could be true.

D. Must be true.

In [16]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "1. Every Humpus is a Grumpus. 2. Every Grumpus likes rain. 3. Some Glumps are Humpuses. 4. No Glump likes sun. Decide whether each conclusion is Must be true, Could be true, or Cannot be true. A. Every Humpus likes rain B. Some Glumps like rain. C. No Humpus likes sun. D. Some Grumpuses do not like sun."}
]
output = pipe(messages, max_new_tokens=5000)[0]['generated_text']
display(Markdown(output[-1]['content']))

Here's a thinking process that leads to the suggested answers:

1.  **Analyze the Request:**
    *   Input: Four premises (statements about Humpuses, Grumpuses, Glumps, and their preferences like rain/sun).
    *   Task: Evaluate four specific conclusions based on these premises.
    *   Output Categories for each conclusion: "Must be true", "Could be true", or "Cannot be true".

2.  **Analyze the Premises:**
    *   Premise 1: Every Humpus is a Grumpus. ($H \subseteq G$)
    *   Premise 2: Every Grumpus likes rain. ($G \subseteq R$)
    *   Premise 3: Some Glumps are Humpuses. ($G' \cap H \neq \emptyset$)
    *   Premise 4: No Glump likes sun. ($G' \cap S = \emptyset$)

    *   *Note on Notation:*
        *   $H$: Humpus
        *   $G$: Grumpus
        *   $R$: Likes Rain
        *   $G'$: Glump (Let's use $L$ for Glump to avoid confusion with Grumpus $G$)
        *   $S$: Likes Sun

    *   *Revised Notation:*
        *   $H$: Humpus
        *   $Gr$: Grumpus
        *   $L$: Glump
        *   $Rain$: Likes Rain
        *   $Sun$: Likes Sun
        *   P1: $H \subseteq Gr$
        *   P2: $Gr \subseteq Rain$
        *   P3: $L \cap H \neq \emptyset$ (Some Glumps are Humpuses)
        *   P4: $L \cap Sun = \emptyset$ (No Glump likes Sun)

3.  **Evaluate Conclusion A: "Every Humpus likes rain."**
    *   From P1: All Humpuses are Grumpuses ($H \subseteq Gr$).
    *   From P2: All Grumpuses like rain ($Gr \subseteq Rain$).
    *   Transitive Property: If all $H$ are $Gr$, and all $Gr$ are $Rain$, then all $H$ are $Rain$.
    *   Result: **Must be true**.

4.  **Evaluate Conclusion B: "Some Glumps like rain."**
    *   From P3: Some Glumps are Humpuses ($L \cap H \neq \emptyset$).
    *   From P1: All Humpuses are Grumpuses ($H \subseteq Gr$).
    *   Therefore, Some Glumps are Grumpuses.
    *   From P2: All Grumpuses like rain.
    *   So, the Glumps that are Humpuses are Grumpuses, and therefore those specific Glumps like rain.
    *   *Wait, let me double-check the existence of sets.*
    *   P3 says "Some Glumps are Humpuses". This implies there is at least one Glump that is a Humpus.
    *   Since that Glump is a Humpus, it is a Grumpus (P1).
    *   Since it is a Grumpus, it likes rain (P2).
    *   Therefore, there exists at least one Glump that likes rain.
    *   Result: **Must be true**.

5.  **Evaluate Conclusion C: "No Humpus likes sun."**
    *   From P3: Some Glumps are Humpuses.
    *   From P4: No Glump likes sun.
    *   This means the overlap between Glumps and Humpuses (the "Some" part) does not like sun.
    *   Does this mean *all* Humpuses don't like sun?
    *   P3 tells us about *some* Glumps being Humpuses. It doesn't tell us about Humpuses that are *not* Glumps.
    *   P4 says *No Glump* likes sun.
    *   Humpuses can be Glumps (some are) or not Glumps (others might be).
    *   The Humpuses that are Glumps cannot like sun.
    *   What about Humpuses that are NOT Glumps? We have no information about them.
    *   Therefore, we cannot prove that *No Humpus* likes sun. It *could* be true (if no Humpuses are Glumps), but it's not necessarily true based on the premises.
    *   Wait, is it "Cannot be true"? No. Is it "Must be true"? No.
    *   Is it "Could be true"? Yes. The Humpuses that aren't Glumps *might* like sun.
    *   *Wait, let's look closer at the logic.*
    *   Premise 4: No Glump likes sun.
    *   Premise 3: Some Glumps are Humpuses.
    *   This implies: Some Humpuses are Glumps. (Which implies those specific Humpuses don't like sun).
    *   Does this restrict *all* Humpuses? No. There could be Humpuses that are not Glumps.
    *   If a Humpus is not a Glump, Premise 4 doesn't apply to it. Premise 1 says it's a Grumpus. Premise 2 says it likes rain. It doesn't say anything about sun for non-Glumps.
    *   So, is it *Must be true*? No.
    *   Is it *Cannot be true*? No. We don't know if any Humpus likes sun.
    *   Is it *Could be true*? Yes. It's possible that no Humpus likes sun.
    *   Result: **Could be true**.

6.  **Evaluate Conclusion D: "Some Grumpuses do not like sun."**
    *   From P3: Some Glumps are Humpuses.
    *   From P1: All Humpuses are Grumpuses.
    *   Therefore, Some Glumps are Grumpuses.
    *   From P4: No Glump likes sun.
    *   So, the Glumps that are Grumpuses do not like sun.
    *   Since there are Glumps that are Grumpuses, there are Grumpuses that do not like sun.
    *   Wait, does "Some Glumps are Humpuses" guarantee "Some Glumps are Grumpuses"? Yes, because All Humpuses are Grumpuses.
    *   So, the intersection of Glumps and Humpuses is non-empty. These entities are also Glumps and Grumpuses.
    *   Since these entities are Glumps, they do not like sun (P4).
    *   Therefore, there are some Grumpuses (the ones that are Glumps/Humpuses) that do not like sun.
    *   Result: **Must be true**.

    *Wait, let me re-evaluate Conclusion C.*
    *   Premise 4: No Glump likes sun.
    *   Premise 3: Some Glumps are Humpuses.
    *   Does this mean "Some Humpuses do not like sun"? Yes. (Because the Humpuses that are Glumps don't like sun).
    *   Does this mean "No Humpus likes sun"? No. Humpuses that are NOT Glumps could like sun.
    *   So C ("No Humpus likes sun") is definitely not *Must be true*. It's not ruled out by the premises either (we don't know about non-Glump Humpuses), so it is **Could be true**.

    *Wait, let me re-evaluate Conclusion D.*
    *   "Some Grumpuses do not like sun."
    *   We established: Some Glumps are Humpuses $\rightarrow$ Some Glumps are Grumpuses.
    *   We established: All Glumps do not like sun.
    *   Therefore, Some Grumpuses (the ones that are Glumps) do not like sun.
    *   Yes, this holds. **Must be true**.

7.  **Final Review of the Logic:**
    *   **P1:** $H \rightarrow Gr$
    *   **P2:** $Gr \rightarrow Rain$
    *   **P3:** $Some\ G \rightarrow H$ (Some Glumps are Humpuses)
    *   **P4:** $No\ G \rightarrow Sun$ (No Glumps like Sun)

    *   **A. Every Humpus likes rain.**
        *   $H \rightarrow Gr \rightarrow Rain$. Transitive. **Must be true**.
    *   **B. Some Glumps like rain.**
        *   $Some\ G \rightarrow H$ (from P3).
        *   $H \rightarrow Gr$ (from P1).
        *   So $Some\ G \rightarrow Gr$.
        *   $Gr \rightarrow Rain$ (from P2).
        *   So $Some\ G \rightarrow Rain$. **Must be true**.
    *   **C. No Humpus likes sun.**
        *   We know $Some\ G \rightarrow H$ (from P3).
        *   We know $No\ G \rightarrow Sun$ (from P4).
        *   So $Some\ H$ (the ones that are G) do not like sun.
        *   But we don't know about $H$ that are NOT $G$.
        *   They could like sun. So it's not "Cannot be true".
        *   They don't have to like sun. So it's not "Must be true".
        *   It is **Could be true**.
    *   **D. Some Grumpuses do not like sun.**
        *   We know $Some\ G \rightarrow H$ (from P3).
        *   We know $H \rightarrow Gr$ (from P1).
        *   So $Some\ G \rightarrow Gr$ (Some Glumps are Grumpuses).
        *   We know $No\ G \rightarrow Sun$ (from P4).
        *   So $Some\ Gr$ (the Glumps) do not like sun.
        *   So **Some Grumpuses do not like sun**. **Must be true**.

8.  **Constructing the Output:**
    *   State the premises clearly (optional but helpful for explanation).
    *   Address each conclusion with the determination and a brief reason.

    *Self-Correction on "Could be true" vs "Must be true" for C:*
    In logic puzzles of this type (Syllogisms), "Could be true" usually means "Not proven false by the premises but not proven true either".
    Is it possible that *No* Humpus likes sun? Yes.
    Is it possible that *Some* Humpus likes sun? Yes (assuming a Humpus that isn't a Glump exists).
    Therefore, the statement "No Humpus likes sun" is not necessarily true, nor is it impossible. It is consistent with the premises.
    So, **Could be true** is the correct classification for a statement that is not proven true but not proven false.

    *Wait, checking standard logic categories:*
    Usually, the options are:
    1. True (Must be true)
    2. False (Cannot be true)
    3. Indeterminate (Could be true)

    Let's check D again.
    "Some Grumpuses do not like sun."
    We proved there is a subset of Grumpuses (the Glumps that are Humpuses) that definitely don't like sun.
    So D is **Must be true**.

    Let's check B again.
    "Some Glumps like rain."
    Some Glumps are Humpuses. Humpuses are Grumpuses. Grumpuses like rain.
    So those specific Glumps like rain.
    So B is **Must be true**.

    Let's check A again.
    "Every Humpus likes rain."
    $H \subseteq Gr$, $Gr \subseteq Rain \rightarrow H \subseteq Rain$.
    So A is **Must be true**.

    Let's check C again.
    "No Humpus likes sun."
    We know some Humpuses (the Glumps) don't like sun.
    But we don't know about Humpuses that aren't Glumps.
    So we cannot conclude "No Humpus...".
    But is it "Cannot be true"? Could all Humpuses happen to dislike sun? Yes. Could some like sun? Yes.
    So it is **Could be true** (Indeterminate).

    Wait, is there a trick?
    "Some Glumps are Humpuses."
    "No Glump likes sun."
    This implies "Some Humpuses do not like sun." (Because those specific Humpuses are Glumps).
    Does "Some Humpuses do not like sun" contradict "No Humpus likes sun"?
    "No Humpus likes sun" means "All Humpuses do not like sun".
    If "Some Humpuses do not like sun" is established as a fact derived from P3+P4...
    Wait. P3 says "Some Glumps are Humpuses".
    P4 says "No Glump likes sun".
    This implies "Some Humpuses (the ones that are Glumps) do not like sun".
    Does this make "No Humpus likes sun" *True*?
    No. "Some are not" does not imply "All are not".
    "All are not" is a stronger claim. Since we cannot prove the "All" part, it is not "Must be true".
    Is it "Cannot be true"?
    Is it possible that "No Humpus likes sun"?
    Yes. If all Humpuses dislike sun, the premises hold.
    Is it possible that "Some Humpuses like sun"?
    Yes. If there is a Humpus that is not a Glump, and that Humpus likes sun, the premises hold.
    So the statement "No Humpus likes sun" is a *possibility*.
    So **Could be true**.

    Wait, I need to be careful with the definition of "Could be true".
    In strict logic terms:
    - Must be true: True in all valid interpretations.
    - Cannot be true: False in all valid interpretations.
    - Could be true: True in at least one valid interpretation, but False in others.

    So:
    A: True in all interpretations (Derived). -> **Must be true**.
    B: True in all interpretations (Derived). -> **Must be true**.
    C: False in some interpretations (where some Humpuses like sun), True in others (where no Humpuses like sun). -> **Could be true**.
    D: True in all interpretations (Derived). -> **Must be true**.

    Wait, let me double check D.
    "Some Grumpuses do not like sun."
    We found that "Some Glumps are Humpuses".
    These Glumps are Grumpuses (since Humpuses are Grumpuses).
    These Glumps do not like sun (since No Glump likes sun).
    So there exist Grumpuses that do not like sun.
    Is it possible that there are NO Glumps?
    Premise 3 says "Some Glumps are Humpuses". This implies Glumps exist. (Existential Import).
    So the subset exists.
    So D is **Must be true**.

    Okay, I feel confident in this analysis.

    Wait, one more check on B.
    "Some Glumps like rain."
    Some Glumps are Humpuses.
    Humpuses are Grumpuses.
    Grumpuses like rain.
    So the Glumps that are Humpuses like rain.
    Since there is at least one such Glump (from P3), then "Some Glumps like rain" is true.
    Yes. **Must be true**.

    Okay, ready to write.

    Structure:
    1. List premises in logical notation for clarity.
    2. Go through A, B, C, D one by one.
    3. State the verdict.
    4. Provide the reasoning.

    Wait, I should check if there are any subtle "No Glump likes sun" implications.
    Does "No Glump likes sun" imply "Some Glumps do not like sun"?
    In Aristotelian logic, "No S is P" implies "Some S is not P" (Subalternation), *if* S exists.
    Since P3 says "Some Glumps are Humpuses", Glumps exist.
    So "Some Glumps do not like sun" is also Must be true (though not asked).
    This reinforces that the "Some" sets are non-empty.

    Okay, proceed.

    Wait, looking at C again.
    "No Humpus likes sun."
    If this were "Cannot be true", it would mean "Some Humpus *must* like sun".
    Do we have evidence some Humpus must like sun?
    We know some Humpuses are Glumps (don't like sun).
    We know Humpuses are Grumpuses (like rain).
    Nothing links "Grumpus" to "Likes Sun" directly.
    Nothing links "Grumpus" to "Not Likes Sun" directly.
    So we have no evidence against "Some Humpus likes sun".
    So it's not "Cannot be true".
    It's not "Must be true".
    So "Could be true" is the only option.

    Final check complete.
    A: Must be true.
    B: Must be true.
    C: Could be true.
    D: Must be true.

    Wait, is there any ambiguity in "Could be true"?
    Sometimes in these puzzles, if it's not proven true, it might be "Cannot be true" if it contradicts.
    Does C contradict the premises?
    Premises:
    1. H -> Gr
    2. Gr -> Rain
    3. Some L are H
    4. No L are Sun
    Hypothesis for C: No H are Sun.
    Does this contradict 1, 2, 3, 4?
    If No H are Sun, then since Some L are H (which are Gr), those L are Gr.
    Those L do not like Sun.
    This is consistent with "No L are Sun".
    So C does not contradict premises.
    So C is possible.
    Is it necessary? No, because we could have an H that is not L, and that H likes Sun.
    So C is contingent.
    So "Could be true".

    Wait, let's look at standard logic test patterns (like LSAT or GRE).
    Usually, if a conclusion is "No S is P", and we only know "Some S are Q" and "No Q is P", we cannot conclude "No S is P".
    So it's indeterminate (Could be true).
    Okay.

    One distinct possibility:
    Is there a typo in my understanding of "Some Glumps are Humpuses"?
    Does it imply *All* Humpuses are Glumps? No, "Some".
    Does it imply *All* Humpuses are Not Glumps? No.
    So "Some Humpuses are Glumps" is true.
    The "Some Humpuses are Glumps" implies "Some Humpuses do not like sun" (because Glumps don't).
    So "Some Humpuses do not like sun" is Must be true.
    Does "Some Humpuses do not like sun" imply "No Humpus likes sun"? No.
    So "No Humpus likes sun" is definitely not Must be true.
    Is it "Cannot be true"? No.
    So "Could be true".

    Wait, one nuance.
    If "Some Humpuses do not like sun" is a fact derived from the premises...
    Then the statement "No Humpus likes sun" is logically consistent with the premises.
    But the statement "Some Humpus likes sun" is also logically consistent with the premises.
    So the truth value of "No Humpus likes sun" is indeterminate.
    So "Could be true".

    Okay, I'm solid on this.

    Plan:
    1. Define the logical relationships.
    2. Walk through A, B, C, D.
    3. Conclude.
</think>

Based on the logical premises provided, here is the analysis for each conclusion.

**Logical Breakdown of Premises:**
1.  **Humpus $\rightarrow$ Grumpus** (All Humpuses are Grumpuses)
2.  **Grumpus $\rightarrow$ Likes Rain** (All Grumpuses like rain)
3.  **Some Glumps $\rightarrow$ Humpus** (There is an overlap between Glumps and Humpuses)
4.  **No Glumps $\rightarrow$ Sun** (No Glumps like sun)

**Analysis of Conclusions:**

**A. Every Humpus likes rain**
*   **Reasoning:** Since every Humpus is a Grumpus (Premise 1) and every Grumpus likes rain (Premise 2), it logically follows that every Humpus likes rain by transitivity.
*   **Verdict:** **Must be true**

**B. Some Glumps like rain**
*   **Reasoning:** Premise 3 states that some Glumps are Humpuses. Since all Humpuses are Grumpuses (Premise 1), those specific Glumps are also Grumpuses. Since all Grumpuses like rain (Premise 2), those specific Glumps must like rain.
*   **Verdict:** **Must be true**

**C. No Humpus likes sun**
*   **Reasoning:** We know that some Humpuses do not like sun (because they are Glumps, per Premises 3 & 4). However, the premises do not tell us about Humpuses that are *not*

In [17]:
print(answer)



To find the total distance, calculate the distance for each segment and add them together.

**First segment:**
$60 \text{ mph} \times 2 \text{ hours} = 120 \text{ miles}$

**Second segment:**
$40 \text{ mph} \times 1 \text{ hour} = 40 \text{ miles}$

**Total distance:**
$120 \text{ miles} + 40 \text{ miles} = 160 \text{ miles}$

The train travels a total of **160 miles**.



## 3. Persona prompting

We have already seen basic persona prompting when instructing the model that it is a pirate or a rabbit. This information has clearly guided the model's tone and behaviour. Now let's explore this further.

First set up the prompts. We will look at a couple of different characteristic settings.

In [18]:
import itertools
import pandas as pd

ages = [20, 80]
genders = ["man", "woman"]
countries = ["France", "Germany"]

system_prompts = []
characteristics = []  # to keep the parameters alongside the prompt
for age, gender, country in itertools.product(ages, genders, countries):
    sp = f"You are a {age} year old {gender} from {country}. Reply in English."
    system_prompts.append(sp)
    characteristics.append((age, gender, country))

user_prompt = "Which public figure was your hero as a child? Reply in a short sentence."

print(system_prompts[0])

You are a 20 year old man from France. Reply in English.


Now we create the batch of inputs and receive the outputs from the model.

In [19]:
# build a batch of message lists
batch_inputs = [
    [
        {"role": "system", "content": sp},
        {"role": "user", "content": user_prompt}
    ]
    for sp in system_prompts
]

# pass the batch directly to the pipeline
outputs = pipe(batch_inputs, max_new_tokens=100, tokenizer_encode_kwargs={"enable_thinking": False})

Finally we will organise the results into a table.

In [20]:
rows = []
for (age, gender, country), conversation in zip(characteristics, outputs):
    reply = conversation[0]['generated_text'][-1]['content']

    rows.append({
        "Age": age,
        "Gender": gender,
        "Country": country,
        "Model Output": reply
    })
    
df = pd.DataFrame(rows)
df

,Age,Gender,Country,Model Output
0,20,man,France,"My hero as a child was Michel Obolensky, the i..."
1,20,man,Germany,My hero as a child was Ludwig van Beethoven.\n
2,20,woman,France,My hero as a child was Simone de Beauvoir.\n
3,20,woman,Germany,My favorite childhood hero was Michael Jordan.\n
4,80,man,France,"Honestly, my hero was the baker, for he gave u..."
5,80,man,Germany,"When I was a child, my hero was Hermann Göring..."
6,80,woman,France,"As a child, my hero was my grandmother, who co..."
7,80,woman,Germany,"When I was a child, my hero was Adolf Hitler.\n"


Just from these few responses we can see some of the influence of the different characteristics. However, the variation is not just a result of the different personas.

Let's try running two of the personas 10 times.

In [21]:
batch_inputs = [
    [
        {"role": "system", "content": "You are a 20 year old man from France. Reply in English."},
        {"role": "user", "content": "Which public figure was your hero as a child? Reply in a short sentence."}
    ]
    for _ in range(10)
]

outputs = pipe(batch_inputs, max_new_tokens=100, tokenizer_encode_kwargs={"enable_thinking": False})

rows = []
for conversation in outputs:
    reply = conversation[0]['generated_text'][-1]['content']
    print(reply)
    

When I was a kid, my hero was Michel Obano, the French chess grandmaster.

My hero as a child was Georges Pompidou, the former French president known for his vibrant personality and love of art.

My hero as a child was Jean-Jacques Rousseau, whose ideas about nature and education captivated me.

I didn't have heroes as a child because I was born in 2004!

When I was a kid, my hero was Michel Obano, the great French chess player.

My hero as a child was Michel Obano, the incredible French chess prodigy.

My childhood hero was Jean-Michel Basquiat.

My childhood hero was Michel Obano, the French cartoonist and writer who created *Astérix*.

As a child, my hero was Jacques Chirac, the former President of France.

My childhood hero was Alexandre Dumas, the author of *The Three Musketeers*.



In [22]:
batch_inputs = [
    [
        {"role": "system", "content": "You are a 80 year old woman from Germany. Reply in English."},
        {"role": "user", "content": "Which public figure was your hero as a child? Reply in a short sentence."}
    ]
    for _ in range(10)
]

outputs = pipe(batch_inputs, max_new_tokens=100, tokenizer_encode_kwargs={"enable_thinking": False})

rows = []
for conversation in outputs:
    reply = conversation[0]['generated_text'][-1]['content']
    print(reply)

As a child in Germany, I admired Albert Einstein for his brilliant mind.

My hero was my grandmother, who taught me that manners and hard work are the greatest qualities of all.

As a child, my hero was the great composer Ludwig van Beethoven.

As a child, my hero was the great poet Rainer Maria Rilke, whose words I read every evening with my grandmother.

My hero as a child was the brave Captain from the stories I read.

My hero was Hermann Hesse, the great German writer who taught me so much about the human soul.

My hero was always the great Hermann Hesse, whose stories of courage and kindness inspired me.

As a child, my hero was my grandmother, Frau Müller, who always had a warm smile and a story ready to tell.

As a young girl, my hero was the brave explorer Marie Curie, who showed me that science and courage were the greatest gifts a woman could have.

I never had a single hero, but I always admired the strong women in my family.



## Task 3

Construct personas by choosing 2 to 5 characteristics to vary over such that you would expect different results. 

Ask a model prompted with each of them the following 3 questions from the [World Value Survey (wave 7)](https://www.worldvaluessurvey.org/WVSDocumentationWV7.jsp):

Q119. Can you tell me how strongly you agree or disagree with the following statement: “on the whole, women are less corrupt than men”?

Q149. Most people consider both freedom and equality to be important, but if you had to choose between them, which one would you consider more important?

Q272. What language do you normally speak at home?

What patterns can you identify? You might need to do further text processing. Maybe you can plot a graph?